# 🔥 Wildfire Mega Dataset — Download & Unification Pipeline
**Proyecto:** Ultralytics Platform × Hospinal Systems  
**Objetivo:** Descargar, estandarizar y unificar datasets públicos de detección de humo y fuego  
**Target:** `/datasets/Wildfire_Mega_Dataset/` — Formato Ultralytics YOLO  
**Clases:** `0: smoke` | `1: fire`

---
###  Datasets incluidos (seleccionados para revisión de licencia)
| # | Dataset | Imágenes | Licencia | Clases |
|

⚠️ **Nota de licencia:** No se incluyen datasets con imágenes de iStockPhoto, Shutterstock o Freepik sin licencia comercial.

In [ ]:
import os
import pandas as pd
from pathlib import Path

# Ruta maestra local
BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()

datasets_folders = [
    "BoWFireDataset",
    "dataset_wildfire.v5i.yolov5pytorch",
    "Fire and Smoke Dataset",
    "Fire Detection from CCTV",
    "FLAME2_RGB",
    "MendeleyDataset"
]

def verify_dataset_integrity(root, folders):
    results = []
    for folder in folders:
        path = root / folder
        results.append({
            "Dataset": folder,
            "Estado": "Accesible" if path.exists() else "No encontrado"
        })
    return pd.DataFrame(results)

df_integrity = verify_dataset_integrity(BASE_PATH, datasets_folders)
print("--- Auditoría de Integridad H'spinal Systems V2.0 ---")
print(df_integrity.to_string(index=False))

In [ ]:
import os
import pandas as pd
from pathlib import Path

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()
datasets = ["BoWFireDataset", "dataset_wildfire.v5i.yolov5pytorch", "Fire and Smoke Dataset",
            "Fire Detection from CCTV", "FLAME2_RGB", "MendeleyDataset"]

def audit_deep_vision(root, folders):
    results = []
    for f in folders:
        p = root / f
        if not p.exists(): continue

        # Diccionario de categorías específicas para investigación
        c = {
            "Dataset": f,
            "Imágenes (RGB)": 0,
            "Videos (.mp4)": 0,
            "Máscaras (GT/Masks)": 0,
            "Etiquetas (YOLO/TXT)": 0,
            "Metadata (CSV/TXT)": 0
        }

        for root_dir, _, files in os.walk(p):
            for file in files:
                name_low = file.lower()
                ext = Path(file).suffix.lower()

                # Lógica de clasificación técnica
                if ext in {'.jpg', '.jpeg', '.png'}:
                    if "gt" in root_dir.lower() or "mask" in root_dir.lower() or "_gt" in name_low:
                        c["Máscaras (GT/Masks)"] += 1
                    else:
                        c["Imágenes (RGB)"] += 1
                elif ext in {'.mp4', '.avi', '.mov'}:
                    c["Videos (.mp4)"] += 1
                elif ext == '.txt' and "readme" not in name_low and "license" not in name_low:
                    c["Etiquetas (YOLO/TXT)"] += 1
                elif ext in {'.csv', '.txt'}:
                    c["Metadata (CSV/TXT)"] += 1

        results.append(c)
    return pd.DataFrame(results)

df_refined = audit_deep_vision(BASE_PATH, datasets)
print("\n--- Auditoría de Visión Computacional Especializada ---")
print(df_refined.to_string(index=False))

### Celda 3 — Inspección profunda de estructura interna

In [ ]:
import os
import pandas as pd
from pathlib import Path

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()
datasets = [
    "BoWFireDataset",
    "dataset_wildfire.v5i.yolov5pytorch",
    "Fire and Smoke Dataset",
    "Fire Detection from CCTV",
    "FLAME2_RGB",
    "MendeleyDataset"
]

def inspect_directory_tree(root, folders, max_depth=3):
    """
    Imprime el árbol de subdirectorios de cada dataset hasta max_depth niveles,
    mostrando conteo de archivos por extensión en cada nivel.
    """
    for folder in folders:
        path = root / folder
        if not path.exists():
            print(f"\n[{folder}] No encontrado en la ruta especificada.")
            continue

        print(f"\n{'='*70}")
        print(f"  DATASET: {folder}")
        print(f"{'='*70}")

        for dirpath, dirnames, filenames in os.walk(path):
            current = Path(dirpath)
            # Calcular profundidad relativa al dataset
            depth = len(current.relative_to(path).parts)
            if depth > max_depth:
                continue

            indent = "  " * depth
            rel = str(current.relative_to(path)) if depth > 0 else "."

            # Contar extensiones en este directorio específico
            ext_count = {}
            for f in filenames:
                ext = Path(f).suffix.lower() or "(sin extensión)"
                ext_count[ext] = ext_count.get(ext, 0) + 1

            # Formatear el resumen de extensiones
            ext_summary = "  |  ".join(
                f"{ext}: {n}" for ext, n in sorted(ext_count.items())
            )

            branch = "└── " if depth > 0 else ""
            print(f"{indent}{branch}{rel}/    ({ext_summary})" if ext_summary else f"{indent}{branch}{rel}/")

# Ejecución de la auditoría de carpetas
inspect_directory_tree(BASE_PATH, datasets)

### Celda 4 — Detección de formato de etiquetas y alerta de origen (Auditoría Forense)


In [ ]:
import yaml
import json
from pathlib import Path
import pandas as pd

def detect_label_format(root, folders):
    """
    Detecta el formato real de las etiquetas inspeccionando el contenido.
    Identifica licencias o metadatos de fuente agregada externa.
    """
    report = []

    for folder in folders:
        path = root / folder
        if not path.exists(): continue

        entry = {
            "Dataset": folder,
            "Formato detectado": "No determinado",
            "Clases": "N/A",
            "Origen fuente agregada externa": "No",
            "Muestra_Contenido": ""
        }

        # 1. Búsqueda de YAML (Indicador de YOLO/fuente agregada externa)
        yaml_files = list(path.rglob("*.yaml"))
        for yf in yaml_files:
            try:
                with open(yf, encoding="utf-8") as f:
                    data = yaml.safe_load(f)
                if isinstance(data, dict):
                    if "roboflow" in str(data).lower():
                        entry["Origen fuente agregada externa"] = "SÍ (Detectado en YAML)"
                    entry["Clases"] = str(data.get("names", data.get("nc", "NC detectado")))
                    entry["Formato detectado"] = "YOLO PyTorch Config"
            except: pass

        # 2. Búsqueda de README para procedencia legal
        for readme in path.rglob("README*"):
            try:
                content = readme.read_text(encoding="utf-8", errors="ignore").lower()
                if "roboflow" in content:
                    entry["Origen fuente agregada externa"] = "SÍ (Mencionado en README)"
            except: pass

        # 3. Inspección binaria/texto de la primera etiqueta encontrada
        # Buscamos archivos .txt que no sean metadatos comunes
        txt_files = [f for f in path.rglob("*.txt") if f.name.lower() not in ["readme.txt", "license.txt", "classes.txt"]]

        if txt_files:
            sample = txt_files[0]
            try:
                with open(sample, 'r', encoding='utf-8', errors='ignore') as f:
                    lines = [f.readline().strip() for _ in range(3)]

                if lines:
                    first_line = lines[0]
                    parts = first_line.split()

                    # Verificación de formato YOLO (Class_ID, X, Y, W, H todos entre 0 y 1)
                    if len(parts) == 5:
                        try:
                            is_yolo = all(0.0 <= float(p) <= 1.0 for p in parts[1:])
                            if is_yolo:
                                entry["Formato detectado"] = "YOLO (Normalizado 0-1)"
                        except ValueError: pass

                    entry["Muestra_Contenido"] = f"Archivo: {sample.name} | Línea: {first_line[:50]}"
            except: pass

        # 4. Caso especial FLAME2 (Metadata de sincronización)
        if folder == "FLAME2_RGB":
            sync_file = list(path.rglob("*Label*"))
            if sync_file:
                entry["Formato detectado"] = "Sincronización Frame-Pair"
                entry["Muestra_Contenido"] = f"Detectado: {sync_file[0].name}"

        report.append(entry)

    return pd.DataFrame(report)

# Ejecución y visualización limpia
df_labels = detect_label_format(BASE_PATH, datasets)
print("\n--- REPORTE TÉCNICO DE ETIQUETADO ---")
print(df_labels.to_string(index=False))

### Celda 5 — Inspeccion de formato XML (Fire and Smoke) y archivo de sincronizacion FLAME2

In [ ]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()


# --- Fire and Smoke Dataset: inspeccion completa de XMLs ---
print("=" * 60)
print("  Fire and Smoke Dataset — Inspeccion VOC XML")
print("=" * 60)

xml_dir = BASE_PATH / "Fire and Smoke Dataset" / "Annotations"
xml_files = sorted(xml_dir.glob("*.xml"))
print(f"Total archivos XML: {len(xml_files)}")

# Recopilar todas las clases presentes en todos los XMLs
all_classes = {}
malformed = 0
for xml_path in xml_files:
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        for obj in root.findall("object"):
            name_el = obj.find("name")
            if name_el is not None and name_el.text:
                name = name_el.text.strip()
                all_classes[name] = all_classes.get(name, 0) + 1
    except ET.ParseError:
        malformed += 1

print(f"Archivos XML malformados: {malformed}")
print("\nClases encontradas en el dataset completo:")
for cls, count in sorted(all_classes.items(), key=lambda x: -x[1]):
    print(f"  '{cls}': {count} instancias")

# Estructura detallada del primer XML como referencia
print(f"\nEstructura del primer XML ({xml_files[0].name}):")
tree = ET.parse(xml_files[0])
root_el = tree.getroot()
print(f"  filename : {root_el.findtext('filename')}")
size = root_el.find("size")
if size is not None:
    print(f"  size     : width={size.findtext('width')}  height={size.findtext('height')}  depth={size.findtext('depth')}")
for i, obj in enumerate(root_el.findall("object")):
    name = obj.find("name").text.strip()
    bbox = obj.find("bndbox")
    xmin = bbox.findtext("xmin")
    ymin = bbox.findtext("ymin")
    xmax = bbox.findtext("xmax")
    ymax = bbox.findtext("ymax")
    print(f"  Objeto {i+1}: class='{name}'  xmin={xmin}  ymin={ymin}  xmax={xmax}  ymax={ymax}")


# --- FLAME2_RGB: inspeccion del archivo Frame Pair Labels ---
print()
print("=" * 60)
print("  FLAME2_RGB — Inspeccion Frame Pair Labels")
print("=" * 60)

flame_dir = BASE_PATH / "FLAME2_RGB"
label_files = [f for f in flame_dir.glob("*.txt")]

for lf in label_files:
    size_kb = lf.stat().st_size / 1024
    print(f"\nArchivo : {lf.name}  ({size_kb:.1f} KB)")
    lines = lf.read_text(encoding="utf-8", errors="ignore").splitlines()
    print(f"Total lineas: {len(lines)}")
    print("\nPrimeras 15 lineas:")
    for i, line in enumerate(lines[:15]):
        print(f"  [{i+1:>3}] {line}")
    print("  ...")
    print("\nUltimas 5 lineas:")
    for line in lines[-5:]:
        print(f"       {line}")

### Celda 6 — Inspeccion de BoWFire (CSV, mascaras GT y carpeta train)

In [ ]:
import sys
!{sys.executable} -m pip install Pillow

In [ ]:
import os
import numpy as np
from pathlib import Path
from PIL import Image

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()
bow_path = BASE_PATH / "BoWFireDataset"


# --- Archivos CSV y TXT en raiz del dataset ---
print("=" * 60)
print("  BoWFireDataset — Archivos de metadatos (CSV / TXT)")
print("=" * 60)

meta_files = list(bow_path.glob("*.csv")) + list(bow_path.glob("*.txt"))
for mf in meta_files:
    size_kb = mf.stat().st_size / 1024
    print(f"\nArchivo: {mf.name}  ({size_kb:.1f} KB)")
    lines = mf.read_text(encoding="utf-8", errors="ignore").splitlines()
    print(f"  Total lineas: {len(lines)}")
    print("  Primeras 5 lineas:")
    for line in lines[:5]:
        print(f"    {line}")


# --- Analisis de mascaras GT ---
print()
print("=" * 60)
print("  BoWFireDataset — Analisis de mascaras GT (dataset/gt/)")
print("=" * 60)

gt_dir   = bow_path / "dataset" / "gt"
img_dir  = bow_path / "dataset" / "img"
masks    = sorted(gt_dir.glob("*.png"))
images   = sorted(img_dir.glob("*.png"))

print(f"Mascaras en dataset/gt/:  {len(masks)}")
print(f"Imagenes en dataset/img/: {len(images)}")

# Verificar correspondencia nombre imagen <-> mascara
print("\nCorrespondencia imagen <-> mascara (primeras 5):")
for mask_path in masks[:5]:
    img_path = img_dir / mask_path.name
    status = "OK" if img_path.exists() else "SIN IMAGEN"
    print(f"  {mask_path.name}  ->  {status}")

# Analisis de valores de pixel de las primeras 3 mascaras
print("\nAnalisis de valores de pixel en mascaras:")
for mask_path in masks[:3]:
    mask = Image.open(mask_path)
    mask_arr = np.array(mask)
    unique_vals = np.unique(mask_arr)
    nonzero = int((mask_arr > 0).sum())
    total = int(mask_arr.size)
    print(f"\n  {mask_path.name}")
    print(f"    Modo: {mask.mode}  |  Dimensiones: {mask.size}")
    print(f"    Valores unicos de pixel: {unique_vals.tolist()}")
    print(f"    Pixels objeto (> 0): {nonzero}  ({nonzero / total * 100:.1f}% del total)")
    print(f"    Pixels fondo  (= 0): {total - nonzero}")


# --- Carpeta train/ ---
print()
print("=" * 60)
print("  BoWFireDataset — Carpeta train/  (240 JPG sin mascaras)")
print("=" * 60)

train_dir = bow_path / "train"
train_imgs = sorted(train_dir.glob("*.jpg"))
print(f"Imagenes en train/: {len(train_imgs)}")

if train_imgs:
    print("Primeros 5 nombres:")
    for img in train_imgs[:5]:
        print(f"  {img.name}")
    print("Ultimos 5 nombres:")
    for img in train_imgs[-5:]:
        print(f"  {img.name}")

### Celda 7 — Conteo definitivo de clases en BoWFire (trainset.csv) y verificacion corregida de mascaras

In [ ]:
import os
import csv
from pathlib import Path

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()
bow_path  = BASE_PATH / "BoWFireDataset"


# --- Conteo de clases en trainset.csv ---
print("=" * 60)
print("  BoWFire — Conteo de clases en trainset.csv")
print("=" * 60)

trainset_csv = bow_path / "trainset.csv"
class_count  = {}
rows_fire    = []
rows_smoke   = []

with open(trainset_csv, encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) < 2:
            continue
        img_path = row[0].strip()
        label    = row[1].strip().lower()
        class_count[label] = class_count.get(label, 0) + 1
        if label == "fire":
            rows_fire.append(img_path)
        elif label == "smoke":
            rows_smoke.append(img_path)

print("\nClases en trainset.csv:")
for cls, count in sorted(class_count.items(), key=lambda x: -x[1]):
    print(f"  '{cls}': {count} imagenes")

print(f"\nPrimeras 5 imagenes fire:")
for p in rows_fire[:5]:
    print(f"  {p}")

print(f"\nPrimeras 5 imagenes smoke:")
for p in rows_smoke[:5]:
    print(f"  {p}")

print(f"\nUltimas 5 imagenes smoke:")
for p in rows_smoke[-5:]:
    print(f"  {p}")


# --- Verificacion corregida de correspondencia mascara <-> imagen ---
# La mascara se llama fire000_gt.png, la imagen se llama fire000.png
# Hay que quitar el sufijo '_gt' antes de buscar la imagen

print()
print("=" * 60)
print("  BoWFire — Correspondencia corregida mascara <-> imagen")
print("=" * 60)

gt_dir  = bow_path / "dataset" / "gt"
img_dir = bow_path / "dataset" / "img"
masks   = sorted(gt_dir.glob("*.png"))

matched   = 0
unmatched = []

for mask_path in masks:
    # Quitar sufijo '_gt' del stem para obtener el nombre de la imagen
    img_name = mask_path.stem.replace("_gt", "") + ".png"
    img_path = img_dir / img_name
    if img_path.exists():
        matched += 1
    else:
        unmatched.append(mask_path.name)

print(f"Mascaras totales : {len(masks)}")
print(f"Con imagen correspondiente: {matched}")
print(f"Sin imagen                : {len(unmatched)}")

if unmatched:
    print("\nMascaras sin imagen (primeras 5):")
    for name in unmatched[:5]:
        print(f"  {name}")
else:
    print("\nCorrespondencia perfecta: todas las mascaras tienen imagen.")


# --- Resumen final de datos disponibles con bbox reales ---
print()
print("=" * 60)
print("  Resumen: Datos con anotaciones de bounding box reales")
print("=" * 60)

summary = [
    ("Fire and Smoke Dataset",  100, "fire",       "VOC XML float coords",   "Convertir VOC -> YOLO"),
    ("BoWFire dataset/img/",    226, "fire",        "Mascara binaria PNG",    "Extraer bbox de mascara -> YOLO"),
]

print(f"\n  {'Dataset':<30} {'Imgs':>5}  {'Clase':<8}  {'Tipo anotacion':<24}  Accion")
print("  " + "-" * 90)
for ds, n, cls, ann_type, action in summary:
    print(f"  {ds:<30} {n:>5}  {cls:<8}  {ann_type:<24}  {action}")

print(f"\n  Total imagenes con bbox reales: {sum(r[1] for r in summary)}")
print(f"  Clase unica con bbox:           fire (clase 1 en el esquema unificado)")
print(f"\n  La clase smoke solo existe como clasificacion (sin bbox):")
print(f"    BoWFire train/ smoke : ~80 imagenes  -> Smart Annotation en Platform")
print(f"    FLAME2 frames (smoke): subset de 53,451 frames -> Smart Annotation")
print(f"    MendeleyDataset fire/ : 950 imagenes -> Smart Annotation")

### Celda 8 — Tabla maestra de decision del pipeline y estructura de salida

In [ ]:
import os
import pandas as pd
from pathlib import Path

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()


# --- Tabla maestra de decision del pipeline ---
print("=" * 70)
print("  DECISION FINAL DEL PIPELINE — Wildfire_Mega_Dataset")
print("=" * 70)

pipeline = [
    # (Dataset, N_imagenes, Clase, Formato_original, Destino, Metodo)
    ("Fire and Smoke Dataset",     100,  "fire",        "VOC XML",         "TRAINING",          "Convertir VOC -> YOLO bbox"),
    ("BoWFire dataset/img/",       226,  "fire",        "Mascara PNG",     "TRAINING",          "Extraer bbox de mascara binaria"),
    ("BoWFire train/ (fire)",      160,  "fire",        "Clasificacion",   "SMART_ANNOTATION",  "Anotar en Ultralytics Platform"),
    ("BoWFire train/ (smoke)",      80,  "smoke",       "Clasificacion",   "SMART_ANNOTATION",  "Anotar en Ultralytics Platform"),
    ("FLAME2_RGB (frames YY/YN)",  "~5k","fire+smoke",  "Video MP4",       "SMART_ANNOTATION",  "Extraer frames -> anotar en Platform"),
    ("MendeleyDataset fire/",      950,  "fire",        "Clasificacion",   "SMART_ANNOTATION",  "Anotar en Ultralytics Platform"),
    ("dataset_wildfire (fuente agregada externa)","--", "--",           "YOLO",           "DESCARTADO",        "Origen fuente agregada externa + iStock (licencia/origen)"),
    ("MendeleyDataset nofire/",    950,  "--",           "Clasificacion",  "DESCARTADO",        "Clase negativa sin valor para deteccion"),
    ("Fire Detection from CCTV",  "~1k","--",            "Sin etiquetas",  "DESCARTADO",        "Sin anotaciones y estructura confusa"),
]

df = pd.DataFrame(pipeline, columns=[
    "Dataset", "Imagenes", "Clase", "Formato original", "Destino", "Metodo"
])

print()
print(df.to_string(index=False))

print()
print("=" * 70)
print("  ESQUEMA DE CLASES UNIFICADO")
print("=" * 70)
print("  Clase 0: smoke  (humo — cualquier densidad)")
print("  Clase 1: fire   (llamas visibles)")
print()
print("  Nota: los datasets de entrenamiento actuales solo aportan")
print("  anotaciones bbox para la clase fire. La clase smoke se")
print("  generara mediante Smart Annotation en Ultralytics Platform.")


# --- Estructura de salida esperada ---
print()
print("=" * 70)
print("  ESTRUCTURA DE SALIDA — Compatible Ultralytics Platform")
print("=" * 70)

output_root = BASE_PATH.parent / "proyect_ultralytics_fires" / "Wildfire_Mega_Dataset"
structure = [
    output_root / "images" / "train",
    output_root / "images" / "val",
    output_root / "images" / "test",
    output_root / "labels" / "train",
    output_root / "labels" / "val",
    output_root / "labels" / "test",
    output_root / "smart_annotation_pool",
    output_root / "logs",
]

print()
for path in structure:
    exists = "(ya existe)" if path.exists() else "(por crear)"
    print(f"  {str(path.relative_to(output_root)): <35}  {exists}")

print()
print("  data.yaml:")
print("    path  : Wildfire_Mega_Dataset/")
print("    train : images/train")
print("    val   : images/val")
print("    test  : images/test")
print("    nc    : 2")
print("    names : [smoke, fire]")


# --- Crear la estructura de directorios ---
print()
print("Creando estructura de directorios...")
for path in structure:
    path.mkdir(parents=True, exist_ok=True)
print("Estructura creada.")


# --- Generar data.yaml ---
import yaml

data_yaml_content = {
    "path":  str(output_root),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    2,
    "names": ["smoke", "fire"],
}

yaml_path = output_root / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml_content, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f"data.yaml generado en: {yaml_path}")

### Celda 9 — Conversion a YOLO: VOC XML (Fire and Smoke) + Mascaras PNG (BoWFire)

In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
import shutil
import random
import json as json_mod
from pathlib import Path
from PIL import Image

random.seed(42)

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("WILDFIRE_OUTPUT_ROOT", "./data/processed/Wildfire_Mega_Dataset")).expanduser().resolve()
LOGS_DIR    = OUTPUT_ROOT / "logs"

# Clase fire = 1 en el esquema unificado [smoke=0, fire=1]
FIRE_CLASS_ID = 1

SPLIT_RATIO = {"train": 0.80, "val": 0.10, "test": 0.10}


def assign_split():
    r = random.random()
    if r < SPLIT_RATIO["train"]:
        return "train"
    elif r < SPLIT_RATIO["train"] + SPLIT_RATIO["val"]:
        return "val"
    else:
        return "test"


def write_yolo_label(label_path, annotations):
    """Escribe un archivo .txt YOLO con una linea por anotacion."""
    with open(label_path, "w", encoding="utf-8") as f:
        for cls_id, xc, yc, w, h in annotations:
            f.write(f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")


# ------------------------------------------------------------------ #
#  CONVERSION 1: Fire and Smoke Dataset  VOC XML -> YOLO              #
# ------------------------------------------------------------------ #
print("=" * 60)
print("  Conversion 1: Fire and Smoke Dataset  (VOC XML -> YOLO)")
print("=" * 60)

img_src_dir = BASE_PATH / "Fire and Smoke Dataset" / "Datacluster Fire and Smoke Sample"
xml_src_dir = BASE_PATH / "Fire and Smoke Dataset" / "Annotations"

stats_voc = {"converted": 0, "skipped": 0, "annotations": 0}

for xml_path in sorted(xml_src_dir.glob("*.xml")):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Dimensiones de imagen desde el XML
        size_el = root.find("size")
        img_w = float(size_el.findtext("width"))
        img_h = float(size_el.findtext("height"))

        if img_w <= 0 or img_h <= 0:
            stats_voc["skipped"] += 1
            continue

        # Buscar imagen correspondiente (mismo stem, extension .jpg)
        img_name = root.findtext("filename")
        if img_name is None:
            img_name = xml_path.stem + ".jpg"
        img_src = img_src_dir / img_name
        if not img_src.exists():
            # Intentar con stem
            candidates = list(img_src_dir.glob(xml_path.stem + ".*"))
            if not candidates:
                stats_voc["skipped"] += 1
                continue
            img_src = candidates[0]

        # Construir anotaciones YOLO
        annotations = []
        for obj in root.findall("object"):
            name_el = obj.find("name")
            if name_el is None or name_el.text is None:
                continue
            cls_name = name_el.text.strip().lower()
            if cls_name != "fire":
                continue
            bbox = obj.find("bndbox")
            xmin = float(bbox.findtext("xmin"))
            ymin = float(bbox.findtext("ymin"))
            xmax = float(bbox.findtext("xmax"))
            ymax = float(bbox.findtext("ymax"))

            # Clamp a limites de imagen
            xmin = max(0.0, min(xmin, img_w))
            ymin = max(0.0, min(ymin, img_h))
            xmax = max(0.0, min(xmax, img_w))
            ymax = max(0.0, min(ymax, img_h))

            if xmax <= xmin or ymax <= ymin:
                continue

            xc = ((xmin + xmax) / 2) / img_w
            yc = ((ymin + ymax) / 2) / img_h
            bw = (xmax - xmin) / img_w
            bh = (ymax - ymin) / img_h
            annotations.append((FIRE_CLASS_ID, xc, yc, bw, bh))

        if not annotations:
            stats_voc["skipped"] += 1
            continue

        # Asignar split y copiar
        split = assign_split()
        prefix = "fas"   # Fire and Smoke
        stem   = f"{prefix}_{xml_path.stem}"

        dst_img = OUTPUT_ROOT / "images" / split / (stem + img_src.suffix)
        dst_lbl = OUTPUT_ROOT / "labels" / split / (stem + ".txt")

        shutil.copy2(img_src, dst_img)
        write_yolo_label(dst_lbl, annotations)

        stats_voc["converted"] += 1
        stats_voc["annotations"] += len(annotations)

    except Exception as e:
        print(f"  Error en {xml_path.name}: {e}")
        stats_voc["skipped"] += 1

print(f"  Imagenes convertidas : {stats_voc['converted']}")
print(f"  Anotaciones escritas : {stats_voc['annotations']}")
print(f"  Omitidas             : {stats_voc['skipped']}")


# ------------------------------------------------------------------ #
#  CONVERSION 2: BoWFire dataset/img/  Mascara PNG -> YOLO bbox       #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Conversion 2: BoWFire dataset  (Mascara PNG -> YOLO bbox)")
print("=" * 60)

gt_dir  = BASE_PATH / "BoWFireDataset" / "dataset" / "gt"
img_dir = BASE_PATH / "BoWFireDataset" / "dataset" / "img"

stats_bow = {"converted": 0, "skipped": 0, "no_fire_region": 0}

for mask_path in sorted(gt_dir.glob("*.png")):
    try:
        # Imagen correspondiente: quitar sufijo _gt del stem
        img_name = mask_path.stem.replace("_gt", "") + ".png"
        img_src  = img_dir / img_name
        if not img_src.exists():
            stats_bow["skipped"] += 1
            continue

        # Cargar mascara y extraer bbox de la region de fuego
        mask = Image.open(mask_path)
        mask_arr = np.array(mask)

        # Mascara RGB binaria: obtener mapa 2D de pixeles con fuego
        if mask_arr.ndim == 3:
            binary = np.any(mask_arr > 0, axis=2)
        else:
            binary = mask_arr > 0

        if not binary.any():
            stats_bow["no_fire_region"] += 1
            continue

        img_h, img_w = binary.shape

        # Bounding box del contorno de la region de fuego
        rows = np.where(np.any(binary, axis=1))[0]
        cols = np.where(np.any(binary, axis=0))[0]
        rmin, rmax = int(rows[0]), int(rows[-1])
        cmin, cmax = int(cols[0]), int(cols[-1])

        xc = ((cmin + cmax) / 2) / img_w
        yc = ((rmin + rmax) / 2) / img_h
        bw = (cmax - cmin) / img_w
        bh = (rmax - rmin) / img_h

        # Clamp por seguridad
        xc = max(0.0, min(1.0, xc))
        yc = max(0.0, min(1.0, yc))
        bw = max(0.001, min(1.0, bw))
        bh = max(0.001, min(1.0, bh))

        annotations = [(FIRE_CLASS_ID, xc, yc, bw, bh)]

        # Asignar split y copiar
        split  = assign_split()
        prefix = "bow"
        stem   = f"{prefix}_{mask_path.stem.replace('_gt', '')}"

        dst_img = OUTPUT_ROOT / "images" / split / (stem + ".png")
        dst_lbl = OUTPUT_ROOT / "labels" / split / (stem + ".txt")

        shutil.copy2(img_src, dst_img)
        write_yolo_label(dst_lbl, annotations)

        stats_bow["converted"] += 1

    except Exception as e:
        print(f"  Error en {mask_path.name}: {e}")
        stats_bow["skipped"] += 1

print(f"  Imagenes convertidas     : {stats_bow['converted']}")
print(f"  Sin region de fuego      : {stats_bow['no_fire_region']}")
print(f"  Omitidas (error/faltante): {stats_bow['skipped']}")


# ------------------------------------------------------------------ #
#  RESUMEN GLOBAL Y LOG                                               #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Resumen de contenido en Wildfire_Mega_Dataset")
print("=" * 60)

total_imgs   = 0
total_labels = 0
for split in ("train", "val", "test"):
    n_imgs = len(list((OUTPUT_ROOT / "images" / split).glob("*")))
    n_lbls = len(list((OUTPUT_ROOT / "labels" / split).glob("*.txt")))
    total_imgs   += n_imgs
    total_labels += n_lbls
    print(f"  {split:<6}  imagenes: {n_imgs:>4}  etiquetas: {n_lbls:>4}")

print(f"\n  Total imagenes : {total_imgs}")
print(f"  Total etiquetas: {total_labels}")
print(f"  Clase presente : fire (id=1)")

# Guardar log JSON
log = {
    "fire_and_smoke_voc": stats_voc,
    "bowfire_masks":      stats_bow,
    "total_images":       total_imgs,
    "total_labels":       total_labels,
    "splits": {
        split: {
            "images": len(list((OUTPUT_ROOT / "images" / split).glob("*"))),
            "labels": len(list((OUTPUT_ROOT / "labels" / split).glob("*.txt"))),
        }
        for split in ("train", "val", "test")
    }
}
log_path = LOGS_DIR / "conversion_log.json"
with open(log_path, "w", encoding="utf-8") as f:
    json_mod.dump(log, f, indent=2)
print(f"\n  Log guardado en: {log_path}")

### Celda 10 — Diagnostico de omisiones, inspeccion de mascaras negras y poblacion del smart_annotation_pool

In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
import shutil
import csv
from pathlib import Path
from PIL import Image

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("WILDFIRE_OUTPUT_ROOT", "./data/processed/Wildfire_Mega_Dataset")).expanduser().resolve()
POOL_DIR    = OUTPUT_ROOT / "smart_annotation_pool"
LOGS_DIR    = OUTPUT_ROOT / "logs"


# ------------------------------------------------------------------ #
#  DIAGNOSTICO 1: Imagen omitida en Fire and Smoke Dataset            #
# ------------------------------------------------------------------ #
print("=" * 60)
print("  Diagnostico: imagen omitida en Fire and Smoke Dataset")
print("=" * 60)

img_src_dir = BASE_PATH / "Fire and Smoke Dataset" / "Datacluster Fire and Smoke Sample"
xml_src_dir = BASE_PATH / "Fire and Smoke Dataset" / "Annotations"

# Identificar cual XML fallo buscando cuales no tienen imagen correspondiente
# o tienen dimensiones invalidas en el XML
for xml_path in sorted(xml_src_dir.glob("*.xml")):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        size_el = root.find("size")
        img_w_str = size_el.findtext("width")  if size_el is not None else None
        img_h_str = size_el.findtext("height") if size_el is not None else None

        # Comprobar si la dimension es invalida o vacia
        if not img_w_str or not img_h_str:
            print(f"  Dimension vacia: {xml_path.name}  w='{img_w_str}'  h='{img_h_str}'")
            continue

        img_w = float(img_w_str)
        img_h = float(img_h_str)
        if img_w <= 0 or img_h <= 0:
            print(f"  Dimension invalida: {xml_path.name}  w={img_w}  h={img_h}")
            continue

        # Verificar imagen existente
        img_name = root.findtext("filename")
        if img_name is None:
            img_name = xml_path.stem + ".jpg"
        img_src = img_src_dir / img_name
        if not img_src.exists():
            candidates = list(img_src_dir.glob(xml_path.stem + ".*"))
            if not candidates:
                print(f"  Imagen no encontrada: {xml_path.name}  -> buscado: {img_name}")

    except ET.ParseError as e:
        print(f"  XML malformado: {xml_path.name}  error: {e}")
    except Exception as e:
        print(f"  Error inesperado: {xml_path.name}  error: {e}")


# ------------------------------------------------------------------ #
#  DIAGNOSTICO 2: Mascaras negras en BoWFire (107 sin region)         #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Diagnostico: mascaras negras en BoWFire (107 sin region)")
print("=" * 60)

gt_dir  = BASE_PATH / "BoWFireDataset" / "dataset" / "gt"
img_dir = BASE_PATH / "BoWFireDataset" / "dataset" / "img"

black_masks     = []
fire_masks      = []

for mask_path in sorted(gt_dir.glob("*.png")):
    mask_arr = np.array(Image.open(mask_path))
    if mask_arr.ndim == 3:
        has_fire = np.any(mask_arr > 0, axis=2).any()
    else:
        has_fire = (mask_arr > 0).any()

    if has_fire:
        fire_masks.append(mask_path)
    else:
        black_masks.append(mask_path)

print(f"Mascaras con fuego (convertidas): {len(fire_masks)}")
print(f"Mascaras negras (sin region):     {len(black_masks)}")

# Verificar el tamano de imagen de algunas mascaras negras
print("\nMuestra de mascaras negras (primeras 5):")
for mp in black_masks[:5]:
    img_name = mp.stem.replace("_gt", "") + ".png"
    img_path = img_dir / img_name
    if img_path.exists():
        img = Image.open(img_path)
        print(f"  {mp.name}  ->  imagen: {img.size[0]}x{img.size[1]}  (existe)")
    else:
        print(f"  {mp.name}  ->  imagen no encontrada")

print(f"\nDecision: las {len(black_masks)} imagenes con mascara negra se")
print("copian al smart_annotation_pool como candidatas a revision manual.")
print("No se les asigna etiqueta. Pueden ser hard negatives o fuego sutil.")


# ------------------------------------------------------------------ #
#  POBLAR smart_annotation_pool                                       #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Poblando smart_annotation_pool")
print("=" * 60)

pool_counts = {}

# --- Subgrupo A: BoWFire mascaras negras (hard negatives potenciales) ---
pool_bow_neg = POOL_DIR / "bowfire_mask_negatives"
pool_bow_neg.mkdir(parents=True, exist_ok=True)
for mp in black_masks:
    img_name = mp.stem.replace("_gt", "") + ".png"
    img_src  = img_dir / img_name
    if img_src.exists():
        shutil.copy2(img_src, pool_bow_neg / img_src.name)
pool_counts["bowfire_mask_negatives"] = len(list(pool_bow_neg.glob("*.png")))

# --- Subgrupo B: BoWFire train/ fire (80 imagenes) ---
pool_bow_fire = POOL_DIR / "bowfire_train_fire"
pool_bow_fire.mkdir(parents=True, exist_ok=True)
bow_train = BASE_PATH / "BoWFireDataset" / "train"
trainset_csv = BASE_PATH / "BoWFireDataset" / "trainset.csv"
with open(trainset_csv, encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) < 2:
            continue
        img_rel = row[0].strip()
        label   = row[1].strip().lower()
        if label == "fire":
            src = BASE_PATH / "BoWFireDataset" / img_rel
            if src.exists():
                shutil.copy2(src, pool_bow_fire / src.name)
pool_counts["bowfire_train_fire"] = len(list(pool_bow_fire.glob("*.jpg")))

# --- Subgrupo C: BoWFire train/ smoke (80 imagenes) ---
pool_bow_smoke = POOL_DIR / "bowfire_train_smoke"
pool_bow_smoke.mkdir(parents=True, exist_ok=True)
with open(trainset_csv, encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) < 2:
            continue
        img_rel = row[0].strip()
        label   = row[1].strip().lower()
        if label == "smoke":
            src = BASE_PATH / "BoWFireDataset" / img_rel
            if src.exists():
                shutil.copy2(src, pool_bow_smoke / src.name)
pool_counts["bowfire_train_smoke"] = len(list(pool_bow_smoke.glob("*.jpg")))

# --- Subgrupo D: MendeleyDataset fire/ (Training and Validation + Testing) ---
pool_mendeley = POOL_DIR / "mendeley_fire"
pool_mendeley.mkdir(parents=True, exist_ok=True)
mendeley_root = BASE_PATH / "MendeleyDataset" / "Dataset"
for subdir in mendeley_root.rglob("fire"):
    if subdir.is_dir():
        for img in subdir.glob("*.jpg"):
            dst = pool_mendeley / img.name
            if not dst.exists():
                shutil.copy2(img, dst)
pool_counts["mendeley_fire"] = len(list(pool_mendeley.glob("*.jpg")))

# --- Resumen ---
print()
total_pool = 0
for subgroup, count in pool_counts.items():
    print(f"  {subgroup:<35}  {count:>4} imagenes")
    total_pool += count

print(f"\n  Total en smart_annotation_pool: {total_pool} imagenes")
print()
print("  Estas imagenes se importan a Ultralytics Platform para")
print("  anotar con Smart Annotation (SAM) y generar la clase smoke.")

# --- Resumen global final ---
print()
print("=" * 60)
print("  Estado final del dataset")
print("=" * 60)

for split in ("train", "val", "test"):
    n_i = len(list((OUTPUT_ROOT / "images" / split).glob("*")))
    n_l = len(list((OUTPUT_ROOT / "labels" / split).glob("*.txt")))
    print(f"  {split:<6}  {n_i:>3} imagenes  {n_l:>3} etiquetas  clase: fire(1)")

print(f"\n  smart_annotation_pool: {total_pool} imagenes sin etiqueta")
print(f"  Siguiente paso: extraer frames de FLAME2 y agregar al pool")

### Celda 11 — Correccion: not_fire como hard negatives en training (no en smart_annotation_pool)

In [ ]:
import os
import shutil
import json as json_mod
from pathlib import Path
from PIL import Image
import numpy as np

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("WILDFIRE_OUTPUT_ROOT", "./data/processed/Wildfire_Mega_Dataset")).expanduser().resolve()
LOGS_DIR    = OUTPUT_ROOT / "logs"

gt_dir  = BASE_PATH / "BoWFireDataset" / "dataset" / "gt"
img_dir = BASE_PATH / "BoWFireDataset" / "dataset" / "img"


# ------------------------------------------------------------------ #
#  PASO 1: Identificar las imagenes not_fire confirmadas              #
# ------------------------------------------------------------------ #
print("=" * 60)
print("  Correccion de pipeline: hard negatives confirmados")
print("=" * 60)

not_fire_masks = sorted([m for m in gt_dir.glob("not_fire*_gt.png")])
print(f"\nMascaras not_fire identificadas: {len(not_fire_masks)}")
print("Todas completamente negras (confirmado en celda anterior).")
print("\nDecision final:")
print("  - NO son candidatas a Smart Annotation (no hay fuego)")
print("  - Se agregan al training set como imagenes sin etiqueta")
print("  - YOLO las interpreta como fondo puro (background)")
print("  - Efecto: reduccion de falsos positivos en el modelo")


# ------------------------------------------------------------------ #
#  PASO 2: Mover not_fire del pool al training set (sin .txt)         #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Moviendo not_fire al training set como hard negatives")
print("=" * 60)

# Eliminar del pool (fueron copiados en celda 10 como bowfire_mask_negatives)
pool_neg_dir = OUTPUT_ROOT / "smart_annotation_pool" / "bowfire_mask_negatives"
if pool_neg_dir.exists():
    shutil.rmtree(pool_neg_dir)
    print(f"  Eliminado de smart_annotation_pool: bowfire_mask_negatives/")

# Crear directorio dedicado para hard negatives (documentacion)
hard_neg_dir = OUTPUT_ROOT / "hard_negatives"
hard_neg_dir.mkdir(parents=True, exist_ok=True)

# Distribuir en splits (misma proporcion 80/10/10, seed fijo)
import random
random.seed(42)

splits_assigned = {"train": 0, "val": 0, "test": 0}
SPLIT_RATIO = {"train": 0.80, "val": 0.10, "test": 0.10}

def assign_split():
    r = random.random()
    if r < SPLIT_RATIO["train"]:
        return "train"
    elif r < SPLIT_RATIO["train"] + SPLIT_RATIO["val"]:
        return "val"
    return "test"

copied = 0
missing = 0
for mask_path in not_fire_masks:
    # Nombre de imagen: not_fire000.png (quitar sufijo _gt)
    img_name = mask_path.stem.replace("_gt", "") + ".png"
    img_src  = img_dir / img_name
    if not img_src.exists():
        missing += 1
        continue

    split  = assign_split()
    prefix = "bow_neg"
    stem   = f"{prefix}_{img_src.stem}"
    dst    = OUTPUT_ROOT / "images" / split / (stem + ".png")

    shutil.copy2(img_src, dst)
    # NO se crea archivo .txt -> YOLO lo trata como fondo puro
    splits_assigned[split] += 1
    copied += 1

    # Copia de referencia en hard_negatives/ para documentacion
    shutil.copy2(img_src, hard_neg_dir / (stem + ".png"))

print(f"\n  Copiadas al training set (sin etiqueta): {copied}")
print(f"  No encontradas: {missing}")
print(f"\n  Distribucion en splits:")
for split, n in splits_assigned.items():
    print(f"    {split}: {n} hard negatives")


# ------------------------------------------------------------------ #
#  PASO 3: Estado actualizado del dataset                             #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Estado actualizado del dataset completo")
print("=" * 60)

grand_total_imgs   = 0
grand_total_labels = 0
for split in ("train", "val", "test"):
    imgs_dir = OUTPUT_ROOT / "images" / split
    lbls_dir = OUTPUT_ROOT / "labels" / split
    n_imgs   = len(list(imgs_dir.glob("*")))
    n_lbls   = len(list(lbls_dir.glob("*.txt")))
    n_hard_neg = n_imgs - n_lbls
    grand_total_imgs   += n_imgs
    grand_total_labels += n_lbls
    print(f"  {split:<6}  {n_imgs:>3} imagenes  {n_lbls:>3} etiquetas  {n_hard_neg:>3} hard_negatives")

print(f"\n  Total imagenes (fire labeled + hard neg): {grand_total_imgs}")
print(f"  Total etiquetas (clase fire=1):           {grand_total_labels}")
print(f"  Hard negatives sin etiqueta:              {grand_total_imgs - grand_total_labels}")

# Smart annotation pool actualizado
pool_total = sum(
    len(list(d.glob("*")))
    for d in (OUTPUT_ROOT / "smart_annotation_pool").iterdir()
    if d.is_dir()
)
print(f"\n  smart_annotation_pool: {pool_total} imagenes")
print(f"    (bowfire_train_fire: 80 | bowfire_train_smoke: 80 | mendeley_fire: 950)")
print(f"\n  hard_negatives/ (documentacion): {len(list(hard_neg_dir.glob('*.png')))}")

print()
print("=" * 60)
print("  Resumen final de clases por fuente")
print("=" * 60)
summary = [
    ("Fire and Smoke Dataset", "fire (1)", "VOC XML -> YOLO",    "~99 imgs"),
    ("BoWFire fire masks",     "fire (1)", "PNG mask -> YOLO",   "119 imgs"),
    ("BoWFire not_fire",       "fondo",    "Hard negative",      "107 imgs, sin .txt"),
    ("BoWFire train fire",     "fire (1)", "Smart Annotation",   "80 imgs en pool"),
    ("BoWFire train smoke",    "smoke (0)","Smart Annotation",   "80 imgs en pool"),
    ("MendeleyDataset fire",   "fire (1)", "Smart Annotation",   "950 imgs en pool"),
    ("FLAME2 frames",          "fire+smoke","Extraer + Anotar",  "pendiente celda 12"),
]
print(f"\n  {'Fuente':<28} {'Clase':<12} {'Metodo':<22} Cantidad")
print("  " + "-" * 75)
for row in summary:
    print(f"  {row[0]:<28} {row[1]:<12} {row[2]:<22} {row[3]}")

In [ ]:
%pip install opencv-python

### Celda 12 — Extraccion de frames FLAME2 RGB (CPU local, sin GPU requerida)

In [ ]:
import os
import cv2
import shutil
import json as json_mod
from pathlib import Path

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("WILDFIRE_OUTPUT_ROOT", "./data/processed/Wildfire_Mega_Dataset")).expanduser().resolve()
FLAME_DIR   = BASE_PATH / "FLAME2_RGB"
LOGS_DIR    = OUTPUT_ROOT / "logs"

# Tasa de muestreo: 1 frame cada SAMPLE_EVERY frames del video
# A ~30fps esto equivale a ~1 imagen por segundo
SAMPLE_EVERY = 30

# Prefijo para identificar frames extraidos de FLAME2
FRAME_PREFIX = "fl2"


# ------------------------------------------------------------------ #
#  PASO 1: Identificar videos RGB e IR                                #
# ------------------------------------------------------------------ #
print("=" * 60)
print("  FLAME2 — Inventario de videos RGB e IR")
print("=" * 60)

all_videos  = sorted(FLAME_DIR.glob("*.MP4")) + sorted(FLAME_DIR.glob("*.mp4"))
rgb_videos  = sorted([v for v in all_videos if "RGB" in v.name])
ir_videos   = sorted([v for v in all_videos if "IR"  in v.name])

print(f"\nVideos RGB: {len(rgb_videos)}")
for v in rgb_videos:
    mb = v.stat().st_size / (1024**2)
    print(f"  {v.name:<40}  {mb:>7.1f} MB")

print(f"\nVideos IR:  {len(ir_videos)}  (solo referencia visual, NO se procesan)")
for v in ir_videos:
    mb = v.stat().st_size / (1024**2)
    print(f"  {v.name:<40}  {mb:>7.1f} MB")


# ------------------------------------------------------------------ #
#  PASO 2: Contar frames por video RGB y construir mapa global        #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Contando frames por video RGB (puede tardar ~1 min)")
print("=" * 60)

video_frame_map = []   # [(video_path, start_global, end_global, fps)]
cumulative = 0

for v in rgb_videos:
    cap = cv2.VideoCapture(str(v))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    start = cumulative + 1
    end   = cumulative + total
    video_frame_map.append((v, start, end, fps, width, height))
    cumulative = end
    print(f"  {v.name:<38}  frames: {total:>6}  fps: {fps:.1f}  {width}x{height}  [{start}-{end}]")

print(f"\n  Total frames RGB acumulados: {cumulative}")
print(f"  Total segun label file:      53451")
if cumulative == 53451:
    print("  Coincidencia perfecta con el archivo de etiquetas.")
else:
    diff = abs(cumulative - 53451)
    print(f"  Diferencia: {diff} frames (posible variacion de codec)")


# ------------------------------------------------------------------ #
#  PASO 3: Parsear el archivo de etiquetas Frame Pair                 #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Parseando archivo de etiquetas Frame Pair Labels")
print("=" * 60)

label_file = FLAME_DIR / "#10) Frame Pair Labels (1).txt"
label_ranges = []   # [(start, end, has_fire, has_smoke)]

with open(label_file, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("Fire") or line.startswith("Smoke") or line.startswith("First"):
            continue
        parts = line.split()
        if len(parts) < 3:
            continue
        try:
            start = int(parts[0])
            end   = int(parts[1])
            flags = parts[2].upper()
            has_fire  = flags[0] == "Y"
            has_smoke = flags[1] == "Y" if len(flags) > 1 else False
            label_ranges.append((start, end, has_fire, has_smoke))
        except (ValueError, IndexError):
            continue

print(f"\nRangos parseados: {len(label_ranges)}")
print(f"\n  {'Start':>7}  {'End':>7}  Fire  Smoke  Frames  Categoria")
print("  " + "-" * 55)
for start, end, fire, smoke, in label_ranges:
    n_frames = end - start + 1
    cat = ("FIRE+SMOKE" if fire and smoke else
           "FIRE_ONLY"  if fire and not smoke else
           "SMOKE_ONLY" if smoke and not fire else
           "NEGATIVE")
    print(f"  {start:>7}  {end:>7}  {'Y' if fire else 'N':>4}  {'Y' if smoke else 'N':>5}  {n_frames:>6}  {cat}")


# ------------------------------------------------------------------ #
#  PASO 4: Extraer frames por categoria                               #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print(f"  Extrayendo frames (1 cada {SAMPLE_EVERY} frames de video)")
print("=" * 60)

# Crear subdirectorios en el pool
pool_dir = OUTPUT_ROOT / "smart_annotation_pool"
categories = {
    "fire_smoke":  pool_dir / "flame2_fire_smoke",
    "fire_only":   pool_dir / "flame2_fire_only",
    "smoke_only":  pool_dir / "flame2_smoke_only",
}
neg_dir = OUTPUT_ROOT / "hard_negatives"

for d in categories.values():
    d.mkdir(parents=True, exist_ok=True)

def get_category(fire, smoke):
    if fire and smoke:  return "fire_smoke"
    if fire:            return "fire_only"
    if smoke:           return "smoke_only"
    return "negative"

def get_label_for_frame(global_frame_num):
    for start, end, fire, smoke in label_ranges:
        if start <= global_frame_num <= end:
            return fire, smoke
    return False, False

frame_counts = {"fire_smoke": 0, "fire_only": 0, "smoke_only": 0, "negative": 0}
global_offset = 0

for v_path, v_start, v_end, fps, width, height in video_frame_map:
    vid_name = v_path.stem.replace(" ", "_").replace("#", "").replace(")", "").replace("(", "")
    cap = cv2.VideoCapture(str(v_path))
    local_frame = 0

    print(f"\n  Procesando: {v_path.name}")
    print(f"    Frames globales {v_start}-{v_end}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        local_frame += 1

        if local_frame % SAMPLE_EVERY != 0:
            continue

        global_frame = global_offset + local_frame
        fire, smoke  = get_label_for_frame(global_frame)
        cat          = get_category(fire, smoke)
        stem         = f"{FRAME_PREFIX}_{vid_name}_f{global_frame:06d}"

        if cat == "negative":
            dst = neg_dir / f"{stem}.jpg"
            cv2.imwrite(str(dst), frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
        else:
            dst = categories[cat] / f"{stem}.jpg"
            cv2.imwrite(str(dst), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])

        frame_counts[cat] += 1

    cap.release()
    global_offset += (v_end - v_start + 1)
    extracted = sum(v for v in frame_counts.values())
    print(f"    Extraidos hasta ahora: {extracted} frames")

print()
print("=" * 60)
print("  Resumen de extraccion FLAME2")
print("=" * 60)
print(f"\n  Categoria         Frames extraidos")
print("  " + "-" * 35)
for cat, count in frame_counts.items():
    print(f"  {cat:<18}  {count:>6}")

print(f"\n  Total extraidos : {sum(frame_counts.values())}")
print(f"  Negativos (hard): {frame_counts['negative']}  -> hard_negatives/")
print(f"  Para anotacion  : {frame_counts['fire_smoke'] + frame_counts['fire_only'] + frame_counts['smoke_only']}  -> smart_annotation_pool/")

# Guardar log
log_flame = {
    "sample_rate": f"1 frame cada {SAMPLE_EVERY}",
    "rgb_videos_processed": len(rgb_videos),
    "ir_videos_skipped": len(ir_videos),
    "frame_counts": frame_counts,
    "total_extracted": sum(frame_counts.values()),
}
with open(LOGS_DIR / "flame2_extraction_log.json", "w") as f:
    json_mod.dump(log_flame, f, indent=2)
print(f"\n  Log guardado en: {LOGS_DIR}/flame2_extraction_log.json")

### Celda 12b — Correccion: videos deduplicados y extraccion FLAME2 corregida

In [ ]:
import cv2
import shutil
import os
import json as json_mod
from pathlib import Path

BASE_PATH = Path(os.environ.get("WILDFIRE_DATA_ROOT", "./data/raw")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("WILDFIRE_OUTPUT_ROOT", "./data/processed/Wildfire_Mega_Dataset")).expanduser().resolve()
FLAME_DIR   = BASE_PATH / "FLAME2_RGB"
LOGS_DIR    = OUTPUT_ROOT / "logs"

SAMPLE_EVERY   = 30
FRAME_PREFIX   = "fl2"
LABEL_MAX_FRAME = 53451   # frame final documentado en el label file


# ------------------------------------------------------------------ #
#  PASO 0: Limpiar frames extraidos en la celda 12 (ejecucion previa) #
# ------------------------------------------------------------------ #
print("=" * 60)
print("  Limpiando extraccion previa de FLAME2")
print("=" * 60)

dirs_to_clean = [
    OUTPUT_ROOT / "smart_annotation_pool" / "flame2_fire_smoke",
    OUTPUT_ROOT / "smart_annotation_pool" / "flame2_fire_only",
    OUTPUT_ROOT / "smart_annotation_pool" / "flame2_smoke_only",
]
hard_neg_dir = OUTPUT_ROOT / "hard_negatives"

flame2_deleted = 0
for d in dirs_to_clean:
    if d.exists():
        files = list(d.glob(f"{FRAME_PREFIX}_*.jpg"))
        for f in files:
            f.unlink()
            flame2_deleted += 1
        print(f"  Eliminados {len(files)} frames de {d.name}/")

# Limpiar hard_negatives de FLAME2
if hard_neg_dir.exists():
    flame2_neg = list(hard_neg_dir.glob(f"{FRAME_PREFIX}_*.jpg"))
    for f in flame2_neg:
        f.unlink()
        flame2_deleted += 1
    print(f"  Eliminados {len(flame2_neg)} hard negatives de FLAME2")

print(f"\n  Total eliminados: {flame2_deleted} frames")


# ------------------------------------------------------------------ #
#  PASO 1: Obtener lista UNICA de videos usando os.listdir            #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Inventario de videos unicos (sin duplicados)")
print("=" * 60)

# os.listdir devuelve nombres unicos en Windows
all_names = [n for n in os.listdir(FLAME_DIR) if n.upper().endswith(".MP4")]
rgb_names = sorted([n for n in all_names if "RGB" in n.upper()])
ir_names  = sorted([n for n in all_names if "IR"  in n.upper() and "RGB" not in n.upper()])

rgb_videos = [FLAME_DIR / n for n in rgb_names]
ir_videos  = [FLAME_DIR / n for n in ir_names]

print(f"\nVideos RGB unicos: {len(rgb_videos)}")
for v in rgb_videos:
    mb = v.stat().st_size / (1024**2)
    print(f"  {v.name:<40}  {mb:>7.1f} MB")

print(f"\nVideos IR unicos (no procesados): {len(ir_videos)}")
for v in ir_videos:
    mb = v.stat().st_size / (1024**2)
    print(f"  {v.name:<40}  {mb:>7.1f} MB")


# ------------------------------------------------------------------ #
#  PASO 2: Contar frames reales por video RGB                         #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Contando frames por video RGB")
print("=" * 60)

video_frame_map = []
cumulative = 0

for v in rgb_videos:
    cap   = cv2.VideoCapture(str(v))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    start = cumulative + 1
    # Limitar end al maximo del label file para el ultimo video
    end   = min(cumulative + total, LABEL_MAX_FRAME)
    video_frame_map.append((v, start, end, fps, w, h))
    cumulative = cumulative + total
    print(f"  {v.name:<38}  frames: {total:>6}  [{start}-{end}]")
    if cumulative >= LABEL_MAX_FRAME:
        print(f"    (frame {LABEL_MAX_FRAME} alcanzado — restantes: ignorados)")
        break

print(f"\n  Rango cubierto por label file: 1 - {LABEL_MAX_FRAME}")


# ------------------------------------------------------------------ #
#  PASO 3: Parsear label file                                         #
# ------------------------------------------------------------------ #
label_file = FLAME_DIR / "#10) Frame Pair Labels (1).txt"
label_ranges = []

with open(label_file, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or not line[0].isdigit():
            continue
        parts = line.split()
        if len(parts) < 3:
            continue
        try:
            start_f = int(parts[0])
            end_f   = int(parts[1])
            flags   = parts[2].upper()
            has_fire  = flags[0] == "Y"
            has_smoke = len(flags) > 1 and flags[1] == "Y"
            label_ranges.append((start_f, end_f, has_fire, has_smoke))
        except (ValueError, IndexError):
            continue

def get_label_for_frame(global_frame_num):
    for start, end, fire, smoke in label_ranges:
        if start <= global_frame_num <= end:
            return fire, smoke
    return False, False

def get_category(fire, smoke):
    if fire and smoke:  return "fire_smoke"
    if fire:            return "fire_only"
    if smoke:           return "smoke_only"
    return "negative"


# ------------------------------------------------------------------ #
#  PASO 4: Extraccion corregida                                       #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print(f"  Extrayendo frames (1 cada {SAMPLE_EVERY} frames de video)")
print("=" * 60)

pool_dir = OUTPUT_ROOT / "smart_annotation_pool"
categories = {
    "fire_smoke":  pool_dir / "flame2_fire_smoke",
    "fire_only":   pool_dir / "flame2_fire_only",
    "smoke_only":  pool_dir / "flame2_smoke_only",
}
for d in categories.values():
    d.mkdir(parents=True, exist_ok=True)
hard_neg_dir.mkdir(parents=True, exist_ok=True)

frame_counts   = {"fire_smoke": 0, "fire_only": 0, "smoke_only": 0, "negative": 0}
global_offset  = 0

for v_path, v_start, v_end, fps, width, height in video_frame_map:
    vid_name = v_path.stem.replace(" ", "_").replace("#", "").replace(")", "").replace("(", "")
    cap      = cv2.VideoCapture(str(v_path))
    local_frame = 0

    print(f"\n  {v_path.name}  [{v_start}-{v_end}]")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        local_frame += 1
        global_frame = global_offset + local_frame

        # No extraer frames mas alla del rango del label file
        if global_frame > LABEL_MAX_FRAME:
            break

        if local_frame % SAMPLE_EVERY != 0:
            continue

        fire, smoke = get_label_for_frame(global_frame)
        cat         = get_category(fire, smoke)
        stem        = f"{FRAME_PREFIX}_{vid_name}_f{global_frame:06d}"

        if cat == "negative":
            dst = hard_neg_dir / f"{stem}.jpg"
        else:
            dst = categories[cat] / f"{stem}.jpg"

        cv2.imwrite(str(dst), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
        frame_counts[cat] += 1

    cap.release()
    global_offset += (v_end - v_start + 1)
    print(f"    Extraidos: fire_smoke={frame_counts['fire_smoke']}  fire_only={frame_counts['fire_only']}  neg={frame_counts['negative']}")


# ------------------------------------------------------------------ #
#  RESUMEN FINAL                                                      #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Resumen corregido de extraccion FLAME2")
print("=" * 60)

for cat, count in frame_counts.items():
    print(f"  {cat:<18}  {count:>4} frames")

total_pool_frames = sum(v for k, v in frame_counts.items() if k != "negative")
print(f"\n  Para Smart Annotation : {total_pool_frames} frames")
print(f"  Hard negatives (NN)   : {frame_counts['negative']} frames")

# Pool total actualizado
pool_total = sum(
    len(list(d.glob("*")))
    for d in pool_dir.iterdir()
    if d.is_dir()
)
print(f"\n  smart_annotation_pool total: {pool_total} imagenes")
print()

# Resumen global del dataset
print("=" * 60)
print("  Estado final del Wildfire_Mega_Dataset")
print("=" * 60)
for split in ("train", "val", "test"):
    n_i = len(list((OUTPUT_ROOT / "images" / split).glob("*")))
    n_l = len(list((OUTPUT_ROOT / "labels" / split).glob("*.txt")))
    print(f"  {split:<6}  {n_i:>3} imagenes  {n_l:>3} con etiqueta  {n_i-n_l:>3} hard negatives")

total_hard_neg = len(list(hard_neg_dir.glob("*.jpg")))
print(f"\n  hard_negatives/ total: {total_hard_neg}  (BoWFire not_fire + FLAME2 NN)")
print(f"  smart_annotation_pool: {pool_total}  imagenes para Platform")
print()
print("  Pipeline de datos completado.")
print("  Siguiente paso: subir a platform.ultralytics")

log = {"sample_rate": SAMPLE_EVERY, "frame_counts": frame_counts, "pool_total": pool_total}
with open(LOGS_DIR / "flame2_extraction_log_corrected.json", "w") as f:
    json_mod.dump(log, f, indent=2)

### Celda 13 — Preparacion de ZIPs para subida a Ultralytics Platform

In [ ]:
import os
import zipfile
import yaml
import json as json_mod
from pathlib import Path

OUTPUT_ROOT = Path(os.environ.get("WILDFIRE_OUTPUT_ROOT", "./data/processed/Wildfire_Mega_Dataset")).expanduser().resolve()
EXPORT_DIR  = OUTPUT_ROOT.parent / "platform_upload"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------ #
#  ZIP 1: Dataset etiquetado — labeled_dataset.zip                    #
#  Formato Ultralytics Hub: images/ + labels/ + data.yaml             #
# ------------------------------------------------------------------ #
print("=" * 60)
print("  Creando ZIP 1: labeled_dataset.zip")
print("  (incluye imagenes con etiquetas fire y hard negatives)")
print("=" * 60)

zip1_path = EXPORT_DIR / "labeled_dataset.zip"
file_count = 0

with zipfile.ZipFile(zip1_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    # data.yaml en raiz del ZIP
    zf.write(OUTPUT_ROOT / "data.yaml", arcname="data.yaml")

    # images/ y labels/ para los tres splits
    for split in ("train", "val", "test"):
        imgs_dir = OUTPUT_ROOT / "images" / split
        lbls_dir = OUTPUT_ROOT / "labels" / split

        for img_path in sorted(imgs_dir.glob("*")):
            arcname = f"images/{split}/{img_path.name}"
            zf.write(img_path, arcname=arcname)
            file_count += 1

        for lbl_path in sorted(lbls_dir.glob("*.txt")):
            arcname = f"labels/{split}/{lbl_path.name}"
            zf.write(lbl_path, arcname=arcname)
            file_count += 1

size_mb = zip1_path.stat().st_size / (1024**2)
print(f"\n  Archivos incluidos : {file_count}")
print(f"  Tamano del ZIP     : {size_mb:.1f} MB")
print(f"  Ruta               : {zip1_path}")

# Verificar contenido del ZIP
with zipfile.ZipFile(zip1_path) as zf:
    names = zf.namelist()
    imgs_train = [n for n in names if n.startswith("images/train")]
    imgs_val   = [n for n in names if n.startswith("images/val")]
    imgs_test  = [n for n in names if n.startswith("images/test")]
    lbls_train = [n for n in names if n.startswith("labels/train")]
    print(f"\n  Verificacion:")
    print(f"    images/train : {len(imgs_train)}")
    print(f"    images/val   : {len(imgs_val)}")
    print(f"    images/test  : {len(imgs_test)}")
    print(f"    labels/train : {len(lbls_train)}  (solo imagenes etiquetadas)")
    print(f"    data.yaml    : incluido")


# ------------------------------------------------------------------ #
#  ZIP 2: Pool de Smart Annotation — smart_annotation_pool.zip        #
#  Solo imagenes, sin etiquetas                                        #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Creando ZIP 2: smart_annotation_pool.zip")
print("  (imagenes sin etiqueta para anotar con SAM en Platform)")
print("=" * 60)

pool_dir  = OUTPUT_ROOT / "smart_annotation_pool"
zip2_path = EXPORT_DIR / "smart_annotation_pool.zip"
pool_count_by_source = {}

with zipfile.ZipFile(zip2_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for subdir in sorted(pool_dir.iterdir()):
        if not subdir.is_dir():
            continue
        imgs = sorted(subdir.glob("*"))
        imgs = [f for f in imgs if f.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        pool_count_by_source[subdir.name] = len(imgs)
        for img in imgs:
            arcname = f"images/{img.name}"
            zf.write(img, arcname=arcname)

size_mb2 = zip2_path.stat().st_size / (1024**2)
total_pool = sum(pool_count_by_source.values())

print(f"\n  Imagenes por fuente:")
for src, n in pool_count_by_source.items():
    print(f"    {src:<35}  {n:>4}")
print(f"\n  Total imagenes en pool : {total_pool}")
print(f"  Tamano del ZIP         : {size_mb2:.1f} MB")
print(f"  Ruta                   : {zip2_path}")


# ------------------------------------------------------------------ #
#  RESUMEN DE ARCHIVOS PARA SUBIR                                     #
# ------------------------------------------------------------------ #
print()
print("=" * 60)
print("  Archivos listos para subir a platform.ultralytics")
print("=" * 60)
print(f"\n  {EXPORT_DIR}")
print(f"\n  1. labeled_dataset.zip     {zip1_path.stat().st_size/(1024**2):.1f} MB")
print(f"     -> Dataset con anotaciones fire (clase 1)")
print(f"     -> 325 imagenes totales, 218 con bbox YOLO")
print()
print(f"  2. smart_annotation_pool.zip  {zip2_path.stat().st_size/(1024**2):.1f} MB")
print(f"     -> {total_pool} imagenes sin etiqueta")
print(f"     -> Usar Smart Annotation en Platform para clase smoke (0) y fire adicional")
print()
print("  Orden de subida recomendado:")
print("    Primero: labeled_dataset.zip  (establece las clases en Platform)")
print("    Segundo: smart_annotation_pool.zip  (extiende el dataset con imagenes sin anotar)")

In [ ]:
import os
from pathlib import Path

# Definimos la ruta exacta que solicitaste
IMG_PATH = Path(os.environ.get("WILDFIRE_SMART_POOL", "./data/exports/smart_annotation_pool/images")).expanduser().resolve()

def count_images(path):
    # Extensiones de imagen estándar
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

    if not path.exists():
        return f"Error: La ruta no existe. Verifica que la carpeta 'images' se haya creado dentro de 'smart_annotation_pool'."

    # Listamos archivos y filtramos por extensión
    files = [f for f in os.listdir(path) if Path(f).suffix.lower() in valid_extensions]
    total = len(files)

    print(f"{'='*60}")
    print(f" AUDITORÍA DE CARPETA: {path.name}")
    print(f"{'='*60}")
    print(f" Ruta completa : {path}")
    print(f" Total imágenes : {total}")
    print(f"{'='*60}")

    if total > 0:
        print(f" Ejemplo del primer archivo: {files[0]}")

    return total

# Ejecutar el conteo
cantidad = count_images(IMG_PATH)